# Rolling Robust Scaling Pipeline

Numba-accelerated rolling robust scaler for online walk-forward backtests.
Maintains a sorted buffer per feature for O(W) median/IQR scaling.

This notebook walks through the pipeline up to the scaling step:
1. Load data
2. Apply robust transforms (diurnal + semantic + winsorization)
3. Generate HAR lag features (for demonstration)
4. Apply rolling robust scaling to the feature matrix

The final cell writes the complete `scaling.py` module to `../src/scaling.py`.

In [ ]:
# ── Setup: clone repo, install deps ──
import os
import sys

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!pip install -q numpy pandas pyarrow numba scikit-learn tqdm

sys.path.insert(0, "/content/harxhar/colab")

In [ ]:
# ── Load data + robust transform on RV ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data("all30min")
adj_rv, baseline = robust_transform(df, "RV", is_target=True)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"\nadj_RV stats:\n{df['adj_RV'].describe()}")

In [ ]:
# ── Generate HAR features (for demonstration) ──
PERIODS_PER_DAY = 48

def resolve_har_lags(max_lag: int = 3125) -> list[int]:
    seq, v = [], 1
    while v <= max_lag:
        seq.append(v)
        v *= 5
    return seq

lags = resolve_har_lags()
feature_names = []
for lag in lags:
    name = f"har_ma_{lag}"
    df[name] = df["adj_RV"].rolling(window=lag, min_periods=1).mean().shift(1)
    feature_names.append(name)

# Drop burn-in NaN rows
max_lag = lags[-1]
df = df.iloc[max_lag:].reset_index(drop=True)

# Extract numpy arrays
X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)

print(f"Feature matrix: {X.shape}")
print(f"Feature names: {feature_names}")
print(f"\nPre-scaling feature stats:")
print(pd.DataFrame(X, columns=feature_names).describe())

In [ ]:
# ── Initialize RollingRobustScaler and inspect stats ──
from src.scaling import RollingRobustScaler

train_win = 500 * PERIODS_PER_DAY  # 500 days
scaler = RollingRobustScaler(train_win, n_features=X.shape[1])
scaler.initialize(X[:train_win])

median, iqr = scaler.get_scaler()
print("Initial scaler stats (from first training window):")
for i, name in enumerate(feature_names):
    print(f"  {name:>15s}  median={median[i]:.4f}  iqr={iqr[i]:.4f}")

# Scale the training window and show before/after
X_scaled_init = (X[:train_win] - median) / iqr

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].boxplot(X[:train_win], labels=feature_names)
axes[0].set_title("Raw features (training window)")
axes[0].tick_params(axis="x", rotation=45)

axes[1].boxplot(X_scaled_init, labels=feature_names)
axes[1].set_title("Scaled features (median/IQR)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ── Convenience methods: transform_single and transform_buffer ──
# transform_single: scale one observation using current window stats
x_test = X[train_win]
x_scaled = scaler.transform_single(x_test)

print("Single observation scaling:")
for i, name in enumerate(feature_names):
    print(f"  {name:>15s}  raw={x_test[i]:.4f}  scaled={x_scaled[i]:.4f}")

# transform_buffer: scale the entire current window at once
X_buf_scaled = scaler.transform_buffer()
print(f"\nBuffer-scaled shape: {X_buf_scaled.shape}")
print(f"Buffer-scaled mean (should be ~0): {X_buf_scaled.mean(axis=0).round(3)}")
print(f"Buffer-scaled std  (should be ~1): {X_buf_scaled.std(axis=0).round(3)}")

# Simulate a few update steps
for t in range(train_win, train_win + 10):
    scaler.update(X[t])

median_after, iqr_after = scaler.get_scaler()
print("\nScaler stats after 10 update steps:")
for i, name in enumerate(feature_names):
    print(f"  {name:>15s}  median={median_after[i]:.4f} (was {median[i]:.4f})  "
          f"iqr={iqr_after[i]:.4f} (was {iqr[i]:.4f})")

In [ ]:
%%writefile ../src/scaling.py
"""Rolling robust scaler for online walk-forward backtests.

Numba-accelerated sorted-matrix quantile tracking for O(W) median/IQR
scaling per feature.  No imports from core/ or projects/.
"""

import numpy as np
from numba import njit

# ---------------------------------------------------------------------------
# Numba kernels
# ---------------------------------------------------------------------------


@njit(cache=True)
def _update_sorted_matrix(sorted_mat: np.ndarray, x_old: np.ndarray, x_new: np.ndarray) -> None:
    """Replace *x_old* with *x_new* in each feature's sorted window."""
    n_features, w = sorted_mat.shape
    for i in range(n_features):
        v_old = x_old[i]
        v_new = x_new[i]
        idx_old = np.searchsorted(sorted_mat[i], v_old)
        idx_new = np.searchsorted(sorted_mat[i], v_new)
        if idx_old < idx_new:
            idx_new -= 1
            for j in range(idx_old, idx_new):
                sorted_mat[i, j] = sorted_mat[i, j + 1]
        elif idx_old > idx_new:
            for j in range(idx_old, idx_new, -1):
                sorted_mat[i, j] = sorted_mat[i, j - 1]
        sorted_mat[i, idx_new] = v_new


@njit(cache=True)
def _get_robust_stats(
    sorted_mat: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute median and IQR from pre-sorted rolling window."""
    n_features, w = sorted_mat.shape
    median = np.empty(n_features, dtype=np.float64)
    iqr = np.empty(n_features, dtype=np.float64)
    idx_25 = (w - 1) * 0.25
    idx_50 = (w - 1) * 0.50
    idx_75 = (w - 1) * 0.75
    i25_floor, rem_25 = int(idx_25), idx_25 - int(idx_25)
    i50_floor, rem_50 = int(idx_50), idx_50 - int(idx_50)
    i75_floor, rem_75 = int(idx_75), idx_75 - int(idx_75)
    for i in range(n_features):
        q25 = sorted_mat[i, i25_floor] * (1.0 - rem_25) + sorted_mat[i, min(i25_floor + 1, w - 1)] * rem_25
        med = sorted_mat[i, i50_floor] * (1.0 - rem_50) + sorted_mat[i, min(i50_floor + 1, w - 1)] * rem_50
        q75 = sorted_mat[i, i75_floor] * (1.0 - rem_75) + sorted_mat[i, min(i75_floor + 1, w - 1)] * rem_75
        median[i] = med
        iq = q75 - q25
        iqr[i] = iq if iq >= 1e-12 else 1.0
    return median, iqr


# ---------------------------------------------------------------------------
# Rolling robust scaler
# ---------------------------------------------------------------------------


class RollingRobustScaler:
    """Online robust scaler backed by sorted-matrix quantile tracking.

    Maintains a rolling window of observations and a parallel sorted matrix
    per feature for O(W) median/IQR computation.  Supports two usage styles:

    1. **get_scaler** -- return (median, iqr) for manual scaling (ridge-style).
    2. **transform_single / transform_buffer** -- scale directly (PCR-style).

    Parameters
    ----------
    window_size : int
        Rolling window length.
    n_features : int, optional
        Number of features.  If given, buffers are pre-allocated;
        otherwise allocation is deferred to :meth:`initialize`.
    """

    def __init__(self, window_size: int, n_features: int | None = None) -> None:
        self.window_size = window_size
        if n_features is not None:
            self.buffer = np.zeros((window_size, n_features), dtype=np.float64)
            self.sorted_mat = np.zeros((n_features, window_size), dtype=np.float64)
        else:
            self.buffer: np.ndarray | None = None  # type: ignore[no-redef]
            self.sorted_mat: np.ndarray | None = None  # type: ignore[no-redef]
        self.pos: int = 0

    def initialize(self, data_block: np.ndarray) -> None:
        """Fill buffers from *data_block* ``(window_size, n_features)``."""
        w = self.window_size
        n_features = data_block.shape[1]
        if self.buffer is None:
            self.buffer = np.empty((w, n_features), dtype=np.float64)
            self.sorted_mat = np.empty((n_features, w), dtype=np.float64)
        self.buffer[:] = data_block[:w]
        for i in range(n_features):
            self.sorted_mat[i] = np.sort(data_block[:w, i])  # type: ignore[index]
        self.pos = 0

    def update(self, x_new: np.ndarray) -> None:
        """Slide window: replace oldest row with *x_new*."""
        assert self.buffer is not None and self.sorted_mat is not None
        x_old = self.buffer[self.pos].copy()
        self.buffer[self.pos] = x_new
        _update_sorted_matrix(self.sorted_mat, x_old, x_new)
        self.pos = (self.pos + 1) % self.window_size

    def get_scaler(self) -> tuple[np.ndarray, np.ndarray]:
        """Return ``(median, iqr)`` arrays from the current sorted buffer."""
        assert self.sorted_mat is not None
        return _get_robust_stats(self.sorted_mat)

    def transform_single(self, x: np.ndarray) -> np.ndarray:
        """Scale a single observation using current median/IQR."""
        median, iqr = self.get_scaler()
        return (x - median) / iqr

    def transform_buffer(self) -> np.ndarray:
        """Scale the entire current buffer using current median/IQR."""
        assert self.buffer is not None
        median, iqr = self.get_scaler()
        return (self.buffer - median) / iqr